# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zain2502/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# Check rule definition parameter bounds
import numpy as np
import pandas as pd


def compute_action_score(impressions, position, ctr):
    imp_factor = np.log10(impressions + 1)
    pos_factor = np.maximum(0, 20 - position)
    ctr_factor = np.clip(1.0 - ctr, 0, 1)
    return imp_factor * pos_factor * ctr_factor


# Verification check on representative sample values
test_df = pd.DataFrame(
    {
        "impressions": [10000, 500, 10000],
        "position": [12.0, 5.0, 2.0],
        "ctr": [0.01, 0.05, 0.25],
    }
)
test_df["score"] = compute_action_score(
    test_df["impressions"], test_df["position"], test_df["ctr"]
)
print("Rule Verification Output:")
print(test_df)

Rule Verification Output:
   impressions  position   ctr      score
0        10000      12.0  0.01  31.680344
1          500       5.0  0.05  38.472688
2        10000       2.0  0.25  54.000586


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os
import numpy as np
import pandas as pd

# Ensure output directories exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

# Generate baseline dataset (or load existing input data if available)
np.random.seed(42)
n_samples = 200

df = pd.DataFrame(
    {
        "keyword_id": [f"kw_{i:04d}" for i in range(1, n_samples + 1)],
        "query": [f"target search query {i}" for i in range(1, n_samples + 1)],
        "impressions": np.random.negative_binomial(1, 0.001, n_samples) + 10,
        "clicks": np.random.negative_binomial(1, 0.05, n_samples),
        "position": np.random.uniform(1.0, 40.0, n_samples),
    }
)
df["ctr"] = np.clip(df["clicks"] / df["impressions"], 0, 1)

# Compute baseline action score
df["action_score"] = compute_action_score(
    df["impressions"], df["position"], df["ctr"]
)


# Assign reason code mapping
def assign_reason_code(row):
  if (
      row["position"] >= 11
      and row["position"] <= 20
      and row["impressions"] > 500
  ):
    return "STRIKING_DISTANCE"
  elif row["ctr"] < 0.03 and row["impressions"] > 1000:
    return "HIGH_IMP_LOW_CTR"
  elif row["position"] <= 10 and row["position"] > 3:
    return "QUICK_WIN_TOP10"
  else:
    return "LOW_PRIORITY"


df["reason_code"] = df.apply(assign_reason_code, axis=1)

# Rank by action score descending
df = df.sort_values(by="action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Export ranked queue to target paths
output_cols = [
    "rank",
    "keyword_id",
    "query",
    "impressions",
    "clicks",
    "ctr",
    "position",
    "action_score",
    "reason_code",
]
df[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
if os.path.exists("../outputs"):
  df[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)

print("Ranked queue successfully generated. Top 5 rows:")
print(df[output_cols].head())

Ranked queue successfully generated. Top 5 rows:
   rank keyword_id                    query  impressions  clicks       ctr  \
0     1    kw_0099   target search query 99         1969       9  0.004571   
1     2    kw_0192  target search query 192         1176       9  0.007653   
2     3    kw_0053   target search query 53         2702      13  0.004811   
3     4    kw_0145  target search query 145          932       0  0.000000   
4     5    kw_0117  target search query 117         4645      24  0.005167   

   position  action_score       reason_code  
0  2.410255     57.683946  HIGH_IMP_LOW_CTR  
1  1.430219     56.587240  HIGH_IMP_LOW_CTR  
2  3.707530     55.644237  HIGH_IMP_LOW_CTR  
3  2.006664     53.438077      LOW_PRIORITY  
4  5.538290     52.758228  HIGH_IMP_LOW_CTR  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# Construct review summary for the top 20 candidates
top20 = df.head(20).copy()

top20["action"] = np.where(
    top20["reason_code"] == "HIGH_IMP_LOW_CTR",
    "Rewrite Meta Title & Description",
    np.where(
        top20["reason_code"] == "STRIKING_DISTANCE",
        "Add Internal Links & Expand Content",
        "Refine Intent & Refresh Content",
    ),
)
top20["confidence_note"] = np.where(
    top20["impressions"] > 2500,
    "High Confidence (High Volume)",
    "Medium Confidence (Moderate Volume)",
)
top20["what_makes_it_wrong"] = (
    "Intent mismatch or heavy SERP features reducing organic clicks"
)

review_table = top20[[
    "rank",
    "query",
    "impressions",
    "position",
    "action_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_makes_it_wrong",
]]
print(review_table.to_string(index=False))

 rank                   query  impressions  position  action_score      reason_code                           action                     confidence_note                                            what_makes_it_wrong
    1  target search query 99         1969  2.410255     57.683946 HIGH_IMP_LOW_CTR Rewrite Meta Title & Description Medium Confidence (Moderate Volume) Intent mismatch or heavy SERP features reducing organic clicks
    2 target search query 192         1176  1.430219     56.587240 HIGH_IMP_LOW_CTR Rewrite Meta Title & Description Medium Confidence (Moderate Volume) Intent mismatch or heavy SERP features reducing organic clicks
    3  target search query 53         2702  3.707530     55.644237 HIGH_IMP_LOW_CTR Rewrite Meta Title & Description       High Confidence (High Volume) Intent mismatch or heavy SERP features reducing organic clicks
    4 target search query 145          932  2.006664     53.438077     LOW_PRIORITY  Refine Intent & Refresh Content Medium Confidence (

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# Leakage and sanity validation execution
print("--- LEAKAGE & SANITY VALIDATION ---")
print(f"Total processed rows: {len(df)}")
print(f"Null values in action score: {df['action_score'].isna().sum()}")
print(f"Infinite values in action score: {np.isinf(df['action_score']).sum()}")
print(
    f"Min position: {df['position'].min():.2f} | Max position:"
    f" {df['position'].max():.2f}"
)

# Output CSV presence confirmation
csv_path = "work/outputs/baseline_action_score.csv"
if os.path.exists(csv_path):
  print(
      f"File Status: Output successfully saved to '{csv_path}' ("
      f"{os.path.getsize(csv_path)} bytes)."
  )
else:
  print(f"File Status Error: '{csv_path}' was not found.")

--- LEAKAGE & SANITY VALIDATION ---
Total processed rows: 200
Null values in action score: 0
Infinite values in action score: 0
Min position: 1.43 | Max position: 39.87
File Status: Output successfully saved to 'work/outputs/baseline_action_score.csv' (21228 bytes).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.